
# Unsupervised U-Net for coherent-noise suppression using the SWAN dataset (synthetic and prestack)

This notebook gives a **minimum but complete runnable example** of using a **U-Net in an unsupervised / internal-learning style** to suppress **coherent seismic noise**:

- **horizontal coherent noise**
- **dipping coherent noise**

The workflow is designed around the **SWAN synthetic prestack** `.npz` format, but it is also made robust to small variations in key names and tensor shapes.

---

## What this notebook demonstrates

We take one clean 2D seismic panel from `SWAN_syn_prestack.npz`, contaminate it with synthetic coherent noise, and then recover the clean image **without using the clean target during training**.

So the training stage sees only the noisy observation

$
y = s + n_c,
$

where:

- $y$ = observed noisy image
- $s$ = unknown clean seismic image
- $n_c$ = coherent noise (horizontal or dipping)

The model is a **single-image U-Net prior**:

$
\hat{s}_\theta = f_\theta(z),
$

where $z$ is a fixed random input and $f_\theta$ is the U-Net.

---

## Key idea

A plain Deep Image Prior can fit seismic structure first and noise later, but coherent noise is often too structured and can also be fitted.  
To make the method more stable for **horizontal** and **dipping** coherent noise, we add **direction-aware frequency–wavenumber regularization**.

We optimize

$\mathcal{L}(\theta) =
\lambda_{\mathrm{rec}} \, \|\hat{s}_\theta - y\|_1
+
\lambda_{\mathrm{sig}} \, \|W_{\mathrm{noise}} \odot \mathcal{F}(\hat{s}_\theta)\|_2^2
+
\lambda_{\mathrm{res}} \, \|(1-W_{\mathrm{noise}})\odot \mathcal{F}(y-\hat{s}_\theta)\|_2^2
+
\lambda_{\mathrm{tv}} \, \mathrm{TV}(\hat{s}_\theta).
$

Interpretation:

- the **reconstruction term** keeps the estimate close to the observed data
- the **signal spectral penalty** discourages the denoised image from containing energy inside the known coherent-noise region in the $f$-$k$ domain
- the **residual spectral penalty** encourages the removed component $y-\hat{s}$ to live mainly inside the coherent-noise region
- the **TV penalty** stabilizes training and suppresses weak random artifacts

This is still **unsupervised**, because no clean target is used in the loss.

---

## Why use SWAN synthetic prestack here?

Because it gives realistic seismic textures and events.  
Even though the notebook is unsupervised during optimization, using a realistic SWAN panel makes the illustrations much more meaningful than using a toy synthetic image.

---

## Notebook features

- robust loader for `SWAN_syn_prestack.npz`
- fallback synthetic gather if the file is not available
- coherent-noise generators for **horizontal** and **dipping** cases
- a compact but expressive **U-Net**
- detailed mathematics and adjustable hyperparameters
- loss curves and intermediate iterations
- side-by-side visual comparisons
- numerical metrics **only for post-training evaluation**, not for training


In [1]:

# ============================================================
# 1. Imports
# ============================================================
import os
import math
import random
import warnings
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)


DEVICE = cuda



## 1. Configuration

The defaults below aim to make the notebook:

- easy to understand
- reasonably fast
- easy to tune

The example is **single-image unsupervised denoising**.  
That means we optimize the U-Net on **one noisy panel at a time**.


In [2]:

# ============================================================
# 2. Configuration
# ============================================================
SWAN_PRESTACK_FILE = "../dataset/SWAN_syn_prestack.npz"   # change if needed

# Which SWAN sample/panel to use
SAMPLE_INDEX = 0
CHANNEL_INDEX = 0   # if data has multiple channels/panels inside one sample

# If the loaded panel is larger than this, optionally crop to speed up the demo
USE_CENTER_CROP = True
CROP_NT = 256
CROP_NX = 256

# Noise scenario to run first: "horizontal" or "dipping"
DEFAULT_SCENARIO = "horizontal"

# Coherent-noise generation
DT = 0.004                # s, illustrative
FMIN = 6.0
FMAX = 45.0
NOISE_AMPLITUDE = 0.10    # relative to clean panel std
HORIZONTAL_X_SMOOTH = 21  # larger -> smoother lateral amplitude variation
DIP_SAMPLES_PER_TRACE = 1.2
DIP_SMOOTH = 17
N_NOISE_COMPONENTS = 1

# U-Net / optimization
Z_CHANNELS = 8
BASE_CHANNELS = 32
NUM_STEPS = 800
LR = 1e-3
SHOW_EVERY = 100

# Loss weights
LAMBDA_REC = 1.0 #1.0->0.0, for coherent noise, we do not make the input and output close
LAMBDA_SIG = 0.12
LAMBDA_RES = 0.12
LAMBDA_TV  = 2e-5

# Direction mask parameters
HORIZONTAL_K_WIDTH = 0.08     # narrow band around kx = 0
DIP_SLOPE_WIDTH    = 0.08     # width around target slope line in normalized f-k space
LOW_FREQ_CUTOFF    = 0.02     # ignore very near-zero temporal frequency

# Optional early stopping heuristic
USE_BEST_BY_LOSS = True

# Visualization
FIGSIZE = (14, 4.5)
CMAP = "seismic"

print("Default scenario:", DEFAULT_SCENARIO)


Default scenario: horizontal



## 2. SWAN prestack loading

Because local copies of SWAN files can differ slightly in key names or tensor organization, the loader below tries a few common cases.

The goal is to extract **one 2D panel** of shape `(nt, nx)`.

### Expected possibilities

The loader can handle several common styles, for example:

- `patches.shape == (N, nt, nx)`
- `patches.shape == (N, C, nt, nx)`
- `patches.shape == (N, C, 1, nt, nx)`

If the file is not available, the notebook falls back to a **toy synthetic seismic panel**, so every section remains runnable.


In [ ]:

# ============================================================
# 3. Data loading
# ============================================================
def center_crop_2d(x, nt, nx):
    t0 = max((x.shape[0] - nt) // 2, 0)
    x0 = max((x.shape[1] - nx) // 2, 0)
    return x[t0:t0 + min(nt, x.shape[0]), x0:x0 + min(nx, x.shape[1])]

def generate_toy_seismic(nt=256, nx=256, seed=7):
    rng = np.random.default_rng(seed)
    t = np.arange(nt)
    x = np.arange(nx)
    T, X = np.meshgrid(t, x, indexing="ij")

    panel = np.zeros((nt, nx), np.float32)

    # Several gently curved and dipping reflection events
    for _ in range(16):
        t0 = rng.uniform(20, nt - 20)
        slope = rng.uniform(-0.25, 0.25)
        curve = rng.uniform(-0.0008, 0.0008)
        amp = rng.uniform(0.4, 1.0) * rng.choice([-1.0, 1.0])
        sigma = rng.uniform(1.5, 3.5)

        center = t0 + slope * (X - nx / 2) + curve * (X - nx / 2) ** 2
        panel += amp * np.exp(-0.5 * ((T - center) / sigma) ** 2)

    # Bandlimit by mild temporal smoothing
    kern = np.array([0.15, 0.7, 0.15], dtype=np.float32)
    for ix in range(nx):
        panel[:, ix] = np.convolve(panel[:, ix], kern, mode="same")

    # Mild lateral smoothing
    kernx = np.array([0.1, 0.8, 0.1], dtype=np.float32)
    for it in range(nt):
        panel[it, :] = np.convolve(panel[it, :], kernx, mode="same")

    panel -= panel.mean()
    panel /= (panel.std() + 1e-6)
    return panel.astype(np.float32)

def try_extract_first_array(npz_obj):
    keys = list(npz_obj.keys())
    for k in keys:
        arr = np.asarray(npz_obj[k])
        if arr.ndim >= 3:
            return k, arr
    raise ValueError(f"No array with ndim >= 3 found. Available keys: {keys}")

def extract_2d_panel(arr, sample_index=0, channel_index=0):
    arr = np.asarray(arr)

    if arr.ndim == 3:
        # (N, nt, nx)
        panel = arr[sample_index]
    elif arr.ndim == 4:
        # likely (N, C, nt, nx)
        panel = arr[sample_index, channel_index]
    elif arr.ndim == 5:
        # likely (N, C, 1, nt, nx)
        panel = arr[sample_index, channel_index, 0]
    else:
        raise ValueError(f"Unsupported array shape: {arr.shape}")

    if panel.ndim != 2:
        raise ValueError(f"Could not reduce to 2D panel. Got shape {panel.shape}")
    return np.asarray(panel, dtype=np.float32)

def load_swan_prestack_panel(npz_file, sample_index=0, channel_index=0):
    if not os.path.exists(npz_file):
        print(f"[INFO] {npz_file} not found. Using fallback toy synthetic panel.")
        return generate_toy_seismic(), {"source": "toy_fallback", "key": None, "shape": None}

    data = np.load(npz_file, allow_pickle=True, mmap_mode="r")
    preferred_keys = ["patches", "data", "x", "X", "samples"]

    arr = None
    key_used = None
    for k in preferred_keys:
        if k in data:
            arr = np.asarray(data[k])
            key_used = k
            break

    if arr is None:
        key_used, arr = try_extract_first_array(data)

    panel = extract_2d_panel(arr, sample_index=sample_index, channel_index=channel_index)
    meta = {"source": npz_file, "key": key_used, "shape": arr.shape}
    return panel, meta

clean_panel = load_swan_prestack_panel(
    SWAN_PRESTACK_FILE,
    sample_index=SAMPLE_INDEX,
    channel_index=CHANNEL_INDEX,
)[0]
meta = load_swan_prestack_panel(
    SWAN_PRESTACK_FILE,
    sample_index=SAMPLE_INDEX,
    channel_index=CHANNEL_INDEX,
)[1]

if USE_CENTER_CROP:
    clean_panel = center_crop_2d(clean_panel, CROP_NT, CROP_NX)

clean_panel = clean_panel.astype(np.float32)
clean_panel -= clean_panel.mean()
clean_panel /= (clean_panel.std() + 1e-6)

print("Loaded panel shape:", clean_panel.shape)
print("Source info:", meta)

plt.figure(figsize=(6, 6))
plt.imshow(clean_panel, cmap=CMAP, aspect="auto", vmin=-2, vmax=2)
plt.title("Base clean seismic panel used for the demo")
plt.xlabel("Trace")
plt.ylabel("Time sample")
plt.colorbar(shrink=0.8)
plt.show()



## 3. Coherent-noise synthesis

To compare **horizontal** and **dipping** coherent-noise suppression, we generate band-limited coherent interference.

### 3.1 Band-limited waveform

We first build a 1D random waveform $w(t)$ and band-limit it in frequency:

$
\tilde{W}(f) = B(f) W(f),
$

where $B(f)$ is a smooth band-pass mask.

### 3.2 Horizontal coherent noise

A simple horizontal-noise model is

$
n_h(t,x) = a(x)\, w(t),
$

where $a(x)$ is a slowly varying lateral amplitude function.

This creates nearly flat events across traces.

### 3.3 Dipping coherent noise

A simple dipping-noise model is

$
n_d(t,x) = a(x)\, w\!\bigl(t - p(x-x_0)\bigr),
$

where $p$ is the dip in samples per trace.

This creates coherent linear noise with slope $p$.

These are simplified models, but they are very useful for testing whether the U-Net can remove **directional coherent noise** without clean labels.


In [ ]:

# ============================================================
# 4. Coherent-noise generation
# ============================================================
def smooth1d(x, k):
    if k <= 1:
        return x
    k = int(k)
    if k % 2 == 0:
        k += 1
    kernel = np.ones(k, dtype=np.float32) / k
    return np.convolve(x, kernel, mode="same")

def fft_bandpass_waveform(nt, dt, fmin, fmax, rng):
    white = rng.standard_normal(nt).astype(np.float32)
    W = np.fft.rfft(white)
    freqs = np.fft.rfftfreq(nt, d=dt)

    band = np.zeros_like(freqs, dtype=np.float32)
    inside = (freqs >= fmin) & (freqs <= fmax)
    band[inside] = 1.0

    # mild cosine taper near edges
    taper = 3.0
    lo = (freqs >= max(0, fmin - taper)) & (freqs < fmin)
    hi = (freqs > fmax) & (freqs <= fmax + taper)
    if lo.any():
        u = (freqs[lo] - (fmin - taper)) / taper
        band[lo] = 0.5 - 0.5 * np.cos(np.pi * u)
    if hi.any():
        u = (freqs[hi] - fmax) / taper
        band[hi] = 0.5 + 0.5 * np.cos(np.pi * u)

    out = np.fft.irfft(W * band, n=nt).astype(np.float32)
    out -= out.mean()
    out /= (out.std() + 1e-6)
    return out

def shift_trace_fractional(trace, shift_samples):
    n = len(trace)
    x = np.arange(n, dtype=np.float32)
    xp = x - shift_samples
    return np.interp(x, xp, trace, left=0.0, right=0.0).astype(np.float32)

def make_horizontal_noise(nt, nx, dt, fmin, fmax, amp=0.1, x_smooth=21, n_components=2, seed=7):
    rng = np.random.default_rng(seed)
    noise = np.zeros((nt, nx), np.float32)

    for _ in range(n_components):
        w = fft_bandpass_waveform(nt, dt, fmin, fmax, rng)
        ax = smooth1d(rng.uniform(0.5, 1.5, size=nx).astype(np.float32), x_smooth)
        ax /= (ax.mean() + 1e-6)
        comp = w[:, None] * ax[None, :]
        noise += comp

    noise -= noise.mean()
    noise /= (noise.std() + 1e-6)
    noise *= amp
    return noise.astype(np.float32)

def make_dipping_noise(nt, nx, dt, fmin, fmax, dip_samples_per_trace=1.2, amp=0.1, x_smooth=17, n_components=2, seed=7):
    rng = np.random.default_rng(seed)
    noise = np.zeros((nt, nx), np.float32)

    for j in range(n_components):
        w = fft_bandpass_waveform(nt, dt, fmin, fmax, rng)
        ax = smooth1d(rng.uniform(0.5, 1.5, size=nx).astype(np.float32), x_smooth)
        ax /= (ax.mean() + 1e-6)
        center = nx / 2 + rng.uniform(-0.1 * nx, 0.1 * nx)
        dip = dip_samples_per_trace * rng.choice([-1.0, 1.0]) if j > 0 else dip_samples_per_trace

        comp = np.zeros((nt, nx), np.float32)
        for ix in range(nx):
            shift = dip * (ix - center)
            comp[:, ix] = ax[ix] * shift_trace_fractional(w, shift)

        noise += comp

    noise -= noise.mean()
    noise /= (noise.std() + 1e-6)
    noise *= amp
    return noise.astype(np.float32)

def make_noisy_case(clean, scenario="horizontal", seed=7):
    nt, nx = clean.shape
    if scenario == "horizontal":
        noise = make_horizontal_noise(
            nt, nx, DT, FMIN, FMAX,
            amp=NOISE_AMPLITUDE,
            x_smooth=HORIZONTAL_X_SMOOTH,
            n_components=N_NOISE_COMPONENTS,
            seed=seed,
        )
    elif scenario == "dipping":
        noise = make_dipping_noise(
            nt, nx, DT, FMIN, FMAX,
            dip_samples_per_trace=DIP_SAMPLES_PER_TRACE,
            amp=NOISE_AMPLITUDE,
            x_smooth=DIP_SMOOTH,
            n_components=N_NOISE_COMPONENTS,
            seed=seed,
        )
    else:
        raise ValueError("scenario must be 'horizontal' or 'dipping'")

    noisy = clean + noise
    return noisy.astype(np.float32), noise.astype(np.float32)

horizontal_noisy, horizontal_noise = make_noisy_case(clean_panel, "horizontal", seed=SEED)
dipping_noisy,    dipping_noise    = make_noisy_case(clean_panel, "dipping", seed=SEED + 1)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, (title, noisy, noise) in enumerate([
    ("Horizontal coherent noise", horizontal_noisy, horizontal_noise),
    ("Dipping coherent noise", dipping_noisy, dipping_noise),
]):
    axes[row, 0].imshow(clean_panel, cmap=CMAP, aspect="auto", vmin=-2, vmax=2)
    axes[row, 0].set_title("Clean panel")
    axes[row, 1].imshow(noise, cmap=CMAP, aspect="auto", vmin=-2, vmax=2)
    axes[row, 1].set_title(title + " only")
    axes[row, 2].imshow(noisy, cmap=CMAP, aspect="auto", vmin=-2, vmax=2)
    axes[row, 2].set_title("Observed = clean + noise")
    for ax in axes[row]:
        ax.set_xlabel("Trace")
        ax.set_ylabel("Time")
plt.tight_layout()
plt.show()



## 4. Frequency–wavenumber direction masks

The coherent noise is directional.  
So we build a mask $W_{\mathrm{noise}}(f,k)$ in the 2D Fourier domain.

### 4.1 Horizontal noise mask

Horizontal events correspond to energy near

$
k \approx 0.
$

So we define a vertical strip around $k=0$.

### 4.2 Dipping noise mask

A dipping event in $t$-$x$ maps approximately to a line in $f$-$k$ space.  
For a target dip, we define a soft band around that line.

The mask is not meant to be exact physics.  
It is a **directional prior** telling the optimizer:

- "energy in this region is likely coherent noise"
- "energy outside this region is more likely useful seismic signal"

That is often enough to make internal learning work much better.


In [ ]:

# ============================================================
# 5. f-k masks
# ============================================================
def build_fk_grids(nt, nx, device="cpu"):
    f = torch.fft.fftfreq(nt, d=1.0, device=device)  # normalized frequency
    k = torch.fft.fftfreq(nx, d=1.0, device=device)  # normalized wavenumber
    Fg, Kg = torch.meshgrid(f, k, indexing="ij")
    return Fg, Kg

def make_horizontal_fk_mask(nt, nx, k_width=0.08, low_freq_cutoff=0.02, device="cpu"):
    Fg, Kg = build_fk_grids(nt, nx, device)
    mask = torch.exp(-0.5 * (Kg / max(k_width, 1e-6)) ** 2)
    mask = mask * (torch.abs(Fg) >= low_freq_cutoff).float()
    mask = mask / (mask.max() + 1e-8)
    return mask

def make_dipping_fk_mask(nt, nx, dip_samples_per_trace=1.2, width=0.08, low_freq_cutoff=0.02, device="cpu"):
    Fg, Kg = build_fk_grids(nt, nx, device)

    # normalized slope line through origin: k = a * f
    # 'a' is not a physical velocity; it is a tunable direction parameter.
    a = float(dip_samples_per_trace) / max(nx / nt, 1e-6)
    line = a * Fg
    dist = Kg - line
    mask = torch.exp(-0.5 * (dist / max(width, 1e-6)) ** 2)
    mask = mask + torch.exp(-0.5 * ((Kg + line) / max(width, 1e-6)) ** 2)  # symmetry
    mask = mask * (torch.abs(Fg) >= low_freq_cutoff).float()
    mask = mask / (mask.max() + 1e-8)
    return mask.clamp(0.0, 1.0)

def show_fk_mask(mask, title):
    plt.figure(figsize=(6, 5))
    plt.imshow(mask.cpu().numpy(), aspect="auto", cmap="magma")
    plt.title(title)
    plt.xlabel("Wavenumber index")
    plt.ylabel("Frequency index")
    plt.colorbar(shrink=0.85)
    plt.show()

nt, nx = clean_panel.shape
horizontal_mask = make_horizontal_fk_mask(nt, nx, HORIZONTAL_K_WIDTH, LOW_FREQ_CUTOFF, DEVICE)
dipping_mask    = make_dipping_fk_mask(nt, nx, DIP_SAMPLES_PER_TRACE, DIP_SLOPE_WIDTH, LOW_FREQ_CUTOFF, DEVICE)

show_fk_mask(horizontal_mask, "Horizontal-noise f-k mask")
show_fk_mask(dipping_mask, "Dipping-noise f-k mask")



## 5. U-Net prior

We use a compact U-Net:

$
\hat{s}_\theta = f_\theta(z),
$

with a fixed random input $z$.

This is a form of **internal learning** or **Deep Image Prior**:

- the network architecture itself acts as a prior
- coherent seismic structure is often fitted earlier and more naturally
- noise needs more iterations to be fitted

The skip connections help preserve reflection continuity while the encoder–decoder structure encourages compact structure.

### Why not train $f_\theta(y)$ directly?

Because that would easily collapse to an identity mapping.  
Using a fixed random input $z$ avoids that trivial solution and forces the network to **generate** the denoised image from its prior structure.


In [ ]:

# ============================================================
# 6. Compact U-Net
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1),
            nn.BatchNorm2d(c_out),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1),
            nn.BatchNorm2d(c_out),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class SmallUNet(nn.Module):
    def __init__(self, in_ch=8, base=32, out_ch=1):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)

        self.pool = nn.AvgPool2d(2)

        self.mid = ConvBlock(base * 4, base * 4)

        self.up3 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(base * 2 + base * 2, base * 2)

        self.up2 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base + base, base)

        self.final = nn.Sequential(
            nn.Conv2d(base, base // 2, 3, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(base // 2, out_ch, 1)
        )

    def forward(self, z):
        e1 = self.enc1(z)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m  = self.mid(self.pool(e3))

        d3 = self.up3(m)
        if d3.shape[-2:] != e2.shape[-2:]:
            d3 = F.interpolate(d3, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d3 = self.dec3(torch.cat([d3, e2], dim=1))

        d2 = self.up2(d3)
        if d2.shape[-2:] != e1.shape[-2:]:
            d2 = F.interpolate(d2, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([d2, e1], dim=1))

        out = self.final(d2)
        return out



## 6. Loss terms in detail

Let

$
\hat{s} = f_\theta(z),
\qquad
\hat{n} = y - \hat{s}.
$

Then the loss is

$
\mathcal{L}
=
\lambda_{\mathrm{rec}} \, \|\hat{s} - y\|_1
+
\lambda_{\mathrm{sig}} \, \|W_{\mathrm{noise}} \odot \mathcal{F}(\hat{s})\|_2^2
+
\lambda_{\mathrm{res}} \, \|(1-W_{\mathrm{noise}})\odot \mathcal{F}(\hat{n})\|_2^2
+
\lambda_{\mathrm{tv}}\, \mathrm{TV}(\hat{s}).
$

### Term 1: reconstruction

$
\mathcal{L}_{\mathrm{rec}} = \|\hat{s} - y\|_1
$

This keeps the estimate close to the observation.

### Term 2: keep coherent-noise directions out of the signal

$
\mathcal{L}_{\mathrm{sig}} = \|W_{\mathrm{noise}} \odot \mathcal{F}(\hat{s})\|_2^2
$

This says: the denoised output should have little energy inside the coherent-noise region.

### Term 3: keep the residual mostly inside the coherent-noise region

$
\mathcal{L}_{\mathrm{res}} = \|(1-W_{\mathrm{noise}})\odot \mathcal{F}(y-\hat{s})\|_2^2
$

This says: once something is removed, it should mostly look like the target coherent noise.

### Term 4: weak TV regularization

$
\mathrm{TV}(\hat{s})
=
\sum_{t,x}
\sqrt{(\partial_t \hat{s})^2 + (\partial_x \hat{s})^2 + \varepsilon^2}
$

This suppresses isolated rough artifacts and stabilizes optimization.

---

## Important practical note

This is not meant to be a perfect physical coherent-noise model.  
It is a **minimum working example** showing how an unsupervised U-Net can be guided by:

- architecture prior
- directional spectral prior
- weak spatial regularization

That combination is often enough to separate coherent noise from reflections in a very interpretable way.


In [ ]:

# ============================================================
# 7. Losses and utilities
# ============================================================
def tv_loss(x, eps=1e-3):
    dt = x[:, :, 1:, :] - x[:, :, :-1, :]
    dx = x[:, :, :, 1:] - x[:, :, :, :-1]
    dt = dt[:, :, :, :-1]
    dx = dx[:, :, :-1, :]
    return torch.sqrt(dt * dt + dx * dx + eps * eps).mean()

def spectral_energy(x):
    # x: [B,1,H,W]
    X = torch.fft.fft2(x[:, 0], norm="ortho")
    return torch.abs(X) ** 2

def snr_db(clean, estimate):
    num = np.sum(clean ** 2)
    den = np.sum((clean - estimate) ** 2) + 1e-12
    return 10.0 * np.log10(num / den)

def plot_three(clean, noisy, den, title_prefix=""):
    vmax = max(np.percentile(np.abs(clean), 99), np.percentile(np.abs(noisy), 99), np.percentile(np.abs(den), 99))
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
    axes[0].imshow(clean, cmap=CMAP, aspect="auto", vmin=-vmax, vmax=vmax)
    axes[0].set_title(f"{title_prefix} clean")
    axes[1].imshow(noisy, cmap=CMAP, aspect="auto", vmin=-vmax, vmax=vmax)
    axes[1].set_title(f"{title_prefix} noisy")
    axes[2].imshow(den, cmap=CMAP, aspect="auto", vmin=-vmax, vmax=vmax)
    axes[2].set_title(f"{title_prefix} denoised")
    axes[3].imshow(noisy - den, cmap=CMAP, aspect="auto", vmin=-vmax, vmax=vmax)
    axes[3].set_title("removed component")
    for ax in axes:
        ax.set_xlabel("Trace")
        ax.set_ylabel("Time")
    plt.tight_layout()
    plt.show()

def plot_trace_overlay(clean, noisy, den, trace_idx=None, title=""):
    if trace_idx is None:
        trace_idx = clean.shape[1] // 2
    plt.figure(figsize=(6, 5))
    plt.plot(clean[:, trace_idx], label="clean", linewidth=2)
    plt.plot(noisy[:, trace_idx], label="noisy", alpha=0.8)
    plt.plot(den[:, trace_idx], label="denoised", linewidth=2)
    plt.gca().invert_yaxis()
    plt.title(f"{title} trace {trace_idx}")
    plt.xlabel("Amplitude")
    plt.ylabel("Time sample")
    plt.legend()
    plt.show()



## 7. Unsupervised optimization loop

For one noisy panel $y$, we optimize a U-Net from scratch.

### Iterative process

At iteration $k$:

1. generate $\hat{s}^{(k)} = f_{\theta^{(k)}}(z)$
2. compute residual $\hat{n}^{(k)} = y - \hat{s}^{(k)}$
3. evaluate the loss
4. update $\theta^{(k)}$ with Adam

Because this is internal learning, the whole method can be seen as:

$
\theta^{(k+1)} = \theta^{(k)} - \eta \nabla_\theta \mathcal{L}(\theta^{(k)}).
$

### Why save intermediate iterations?

Because internal-learning methods are often best understood dynamically:

- early iterations: strong structural prior, underfit noise
- middle iterations: often best denoising
- late iterations: risk of re-fitting coherent noise

So we save snapshots during training for visual inspection.


In [ ]:

# ============================================================
# 8. Unsupervised training on one noisy panel
# ============================================================
@dataclass
class RunResult:
    scenario: str
    clean: np.ndarray
    noisy: np.ndarray
    noise_true: np.ndarray
    denoised: np.ndarray
    removed: np.ndarray
    best_step: int
    history: dict
    snapshots: dict

def run_unsupervised_unet(noisy_np, scenario="horizontal", clean_np=None, noise_true_np=None):
    noisy_np = noisy_np.astype(np.float32)
    H, W = noisy_np.shape

    noisy = torch.from_numpy(noisy_np)[None, None].to(DEVICE)

    if scenario == "horizontal":
        fk_mask = make_horizontal_fk_mask(H, W, HORIZONTAL_K_WIDTH, LOW_FREQ_CUTOFF, DEVICE)
    elif scenario == "dipping":
        fk_mask = make_dipping_fk_mask(H, W, DIP_SAMPLES_PER_TRACE, DIP_SLOPE_WIDTH, LOW_FREQ_CUTOFF, DEVICE)
    else:
        raise ValueError("scenario must be 'horizontal' or 'dipping'")

    fk_mask = fk_mask[None, :, :]  # [1,H,W]

    model = SmallUNet(in_ch=Z_CHANNELS, base=BASE_CHANNELS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    z = torch.randn(1, Z_CHANNELS, H, W, device=DEVICE)

    history = {
        "total": [],
        "rec": [],
        "sig": [],
        "res": [],
        "tv": [],
        "snr": [],
    }
    snapshots = {}
    best = {"loss": float("inf"), "step": -1, "den": None}

    for step in range(1, NUM_STEPS + 1):
        optimizer.zero_grad()

        s_hat = model(z)
        r_hat = noisy - s_hat

        Sf = spectral_energy(s_hat)   # [1,H,W]
        Rf = spectral_energy(r_hat)

        rec = torch.mean(torch.abs(s_hat - noisy))
        sig = torch.mean(fk_mask * Sf)
        res = torch.mean((1.0 - fk_mask) * Rf)
        tv  = tv_loss(s_hat)

        loss = LAMBDA_REC * rec + LAMBDA_SIG * sig + LAMBDA_RES * res + LAMBDA_TV * tv
        loss.backward()
        optimizer.step()

        history["total"].append(float(loss.item()))
        history["rec"].append(float(rec.item()))
        history["sig"].append(float(sig.item()))
        history["res"].append(float(res.item()))
        history["tv"].append(float(tv.item()))

        s_np = s_hat.detach().cpu().numpy()[0, 0]
        if clean_np is not None:
            history["snr"].append(float(snr_db(clean_np, s_np)))
        else:
            history["snr"].append(np.nan)

        if loss.item() < best["loss"]:
            best["loss"] = float(loss.item())
            best["step"] = step
            best["den"] = s_np.copy()

        if step == 1 or step % SHOW_EVERY == 0 or step == NUM_STEPS:
            snapshots[step] = s_np.copy()
            print(
                f"[{scenario:10s}] step {step:4d} | "
                f"loss={loss.item():.5f} rec={rec.item():.5f} "
                f"sig={sig.item():.5f} res={res.item():.5f} tv={tv.item():.5f}"
            )

    den = best["den"] if USE_BEST_BY_LOSS else s_np
    removed = noisy_np - den

    return RunResult(
        scenario=scenario,
        clean=clean_np,
        noisy=noisy_np,
        noise_true=noise_true_np,
        denoised=den,
        removed=removed,
        best_step=best["step"],
        history=history,
        snapshots=snapshots,
    )



## 8. Run one case first

Below we run the notebook on the default scenario.  
Then later we run both scenarios and compare them side by side.


In [ ]:

# ============================================================
# 9. Run the default scenario
# ============================================================
if DEFAULT_SCENARIO == "horizontal":
    example_noisy = horizontal_noisy
    example_noise = horizontal_noise
else:
    example_noisy = dipping_noisy
    example_noise = dipping_noise

example_result = run_unsupervised_unet(
    example_noisy,
    scenario=DEFAULT_SCENARIO,
    clean_np=clean_panel,
    noise_true_np=example_noise,
)

print("Best step:", example_result.best_step)
print("Noisy SNR:   %.2f dB" % snr_db(clean_panel, example_result.noisy))
print("Denoised SNR: %.2f dB" % snr_db(clean_panel, example_result.denoised))

plot_three(clean_panel, example_result.noisy, example_result.denoised, title_prefix=DEFAULT_SCENARIO + " |")
plot_trace_overlay(clean_panel, example_result.noisy, example_result.denoised, title=DEFAULT_SCENARIO + " |")



## 9. Loss curves and intermediate iterations

The next figures are useful for understanding **how the internal-learning solution evolves**.

Look for:

- a fast drop in reconstruction loss
- stabilization of directional spectral losses
- visually cleaner reflection structure in the middle or later iterations


In [ ]:

# ============================================================
# 10. Diagnostics for the default run
# ============================================================
hist = example_result.history

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(hist["total"], label="total")
axes[0].plot(hist["rec"], label="rec")
axes[0].plot(hist["sig"], label="signal fk")
axes[0].plot(hist["res"], label="residual fk")
axes[0].set_title("Loss terms")
axes[0].set_xlabel("Iteration")
axes[0].legend()

axes[1].plot(hist["snr"], label="SNR vs clean (evaluation only)")
axes[1].axvline(example_result.best_step - 1, color="k", linestyle="--", label="best saved step")
axes[1].set_title("Evaluation metric through iterations")
axes[1].set_xlabel("Iteration")
axes[1].legend()
plt.tight_layout()
plt.show()

snap_steps = sorted(example_result.snapshots.keys())
vmax = np.percentile(np.abs(clean_panel), 99)

fig, axes = plt.subplots(1, len(snap_steps), figsize=(3.8 * len(snap_steps), 4.2))
if len(snap_steps) == 1:
    axes = [axes]
for ax, step in zip(axes, snap_steps):
    ax.imshow(example_result.snapshots[step], cmap=CMAP, aspect="auto", vmin=-vmax, vmax=vmax)
    ax.set_title(f"iter {step}")
    ax.set_xlabel("Trace")
    ax.set_ylabel("Time")
plt.tight_layout()
plt.show()



## 10. Compare horizontal and dipping coherent-noise cases

Now we run both cases with the same U-Net idea.

This is useful because in practice a method that handles only flat coherent noise may fail on dipping interference, and vice versa.


In [ ]:

# ============================================================
# 11. Run both scenarios
# ============================================================
result_horizontal = run_unsupervised_unet(
    horizontal_noisy,
    scenario="horizontal",
    clean_np=clean_panel,
    noise_true_np=horizontal_noise,
)

result_dipping = run_unsupervised_unet(
    dipping_noisy,
    scenario="dipping",
    clean_np=clean_panel,
    noise_true_np=dipping_noise,
)

summary = {
    "horizontal": {
        "noisy_snr_db": snr_db(clean_panel, result_horizontal.noisy),
        "denoised_snr_db": snr_db(clean_panel, result_horizontal.denoised),
        "best_step": result_horizontal.best_step,
    },
    "dipping": {
        "noisy_snr_db": snr_db(clean_panel, result_dipping.noisy),
        "denoised_snr_db": snr_db(clean_panel, result_dipping.denoised),
        "best_step": result_dipping.best_step,
    }
}
summary


In [ ]:

# ============================================================
# 12. Side-by-side comparison
# ============================================================
def compare_results(res_h, res_d, clean):
    vmax = np.percentile(np.abs(clean), 99)

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    rows = [
        ("Horizontal", res_h),
        ("Dipping", res_d),
    ]
    for i, (name, res) in enumerate(rows):
        axes[i, 0].imshow(clean, cmap=CMAP, aspect="auto", vmin=-vmax, vmax=vmax)
        axes[i, 0].set_title(f"{name}: clean")
        axes[i, 1].imshow(res.noisy, cmap=CMAP, aspect="auto", vmin=-vmax, vmax=vmax)
        axes[i, 1].set_title(f"{name}: noisy")
        axes[i, 2].imshow(res.denoised, cmap=CMAP, aspect="auto", vmin=-vmax, vmax=vmax)
        axes[i, 2].set_title(f"{name}: denoised")
        axes[i, 3].imshow(res.removed, cmap=CMAP, aspect="auto", vmin=-vmax, vmax=vmax)
        axes[i, 3].set_title(f"{name}: removed")
        for j in range(4):
            axes[i, j].set_xlabel("Trace")
            axes[i, j].set_ylabel("Time")
    plt.tight_layout()
    plt.show()

compare_results(result_horizontal, result_dipping, clean_panel)

print("Horizontal case:")
print("  noisy SNR    = %.2f dB" % snr_db(clean_panel, result_horizontal.noisy))
print("  denoised SNR = %.2f dB" % snr_db(clean_panel, result_horizontal.denoised))
print("  best step    =", result_horizontal.best_step)

print("\nDipping case:")
print("  noisy SNR    = %.2f dB" % snr_db(clean_panel, result_dipping.noisy))
print("  denoised SNR = %.2f dB" % snr_db(clean_panel, result_dipping.denoised))
print("  best step    =", result_dipping.best_step)



## 11. Optional: examine the removed component in the $f$-$k$ domain

This helps confirm whether the removed part is actually concentrated in the target coherent-noise direction.


In [ ]:

# ============================================================
# 13. f-k visualization
# ============================================================
def fft_mag_np(x):
    X = np.fft.fftshift(np.fft.fft2(x, norm="ortho"))
    return np.log1p(np.abs(X))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
items = [
    ("Horizontal noisy", result_horizontal.noisy),
    ("Horizontal removed", result_horizontal.removed),
    ("Horizontal denoised", result_horizontal.denoised),
    ("Dipping noisy", result_dipping.noisy),
    ("Dipping removed", result_dipping.removed),
    ("Dipping denoised", result_dipping.denoised),
]
for ax, (ttl, arr) in zip(axes.ravel(), items):
    ax.imshow(fft_mag_np(arr), aspect="auto", cmap="magma")
    ax.set_title(ttl + " | log |F|")
    ax.set_xlabel("Shifted wavenumber")
    ax.set_ylabel("Shifted frequency")
plt.tight_layout()
plt.show()



## 12. Adjustable parameters and how they affect behavior

### `NOISE_AMPLITUDE`
Controls how strong the synthetic coherent noise is.

- larger value → harder denoising problem

### `DIP_SAMPLES_PER_TRACE`
Controls the dip of the synthetic dipping noise.

- larger value → steeper coherent interference

### `LAMBDA_SIG`
Controls how strongly the denoised output is forced to avoid the coherent-noise direction.

- too small → residual coherent noise may remain
- too large → useful dipping signal may be over-suppressed

### `LAMBDA_RES`
Controls how strongly the removed component is forced to stay inside the coherent-noise region.

- too small → removal may be less directional
- too large → model may over-remove energy

### `LAMBDA_TV`
Controls weak spatial smoothing.

- too small → more speckled artifacts
- too large → reflection over-smoothing

### `NUM_STEPS`
Controls the internal-learning optimization length.

- too short → underfit
- too long → risk of fitting some coherent noise back into the model

---

## Suggested starting values

For horizontal noise:

```python
LAMBDA_SIG = 0.10
LAMBDA_RES = 0.10
HORIZONTAL_K_WIDTH = 0.06
```

For dipping noise:

```python
LAMBDA_SIG = 0.12
LAMBDA_RES = 0.12
DIP_SLOPE_WIDTH = 0.08
```

If useful reflections are being removed, first reduce:

- `LAMBDA_SIG`
- `LAMBDA_RES`

If coherent noise remains too strong, increase them gradually.



## 13. Benefits of a unsupervised learning method

During optimization, the loss uses only:

- the **noisy observation** $y$
- a **directional prior**
- a **weak regularization prior**
- the **U-Net architecture prior**

It does **not** use the clean target $s$.

The clean panel from SWAN is used here only to:

1. generate a controlled noisy example
2. evaluate the result afterward

So the training itself remains unsupervised.

---

## Practical extensions

This minimum example can be extended in several useful directions:

1. **multi-panel internal learning**  
   Optimize over several neighboring SWAN panels at once

2. **learnable blind-spot masking**
   Add Noise2Void-style masking for extra robustness

3. **multi-scale directional masks**
   Use several masks to cover a range of dips instead of one target dip

4. **residual U-Net**
   Let the network predict coherent noise directly:
   $
   \hat{n}_\theta = g_\theta(z), \qquad \hat{s} = y - \hat{n}_\theta
   $

5. **curvelet / seislet / slope-domain penalties**
   Replace the simple $f$-$k$ directional mask by a more geology-aware transform

---

## Brief Conclusion

This notebook is a compact demonstration that an **unsupervised U-Net** can suppress **horizontal** and **dipping** coherent seismic noise by combining:

- a **Deep Image Prior / internal-learning** formulation
- a **direction-aware $f$-$k$ regularization**
- a **weak spatial TV stabilization**

That makes it a useful starting point for more realistic coherent-noise attenuation workflows on SWAN prestack data.
